In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.covariance import MinCovDet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.base import clone

# 模型相關
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

# 設定隨機種子
RANDOM_SEED = 42

# ==============================================================================
# 1. 資料讀取與前處理
# ==============================================================================
print("Loading Data...")
df = pd.read_csv('numeric_log_transformed.csv', index_col='Date', parse_dates=True)

# 刪除不要的欄位
cols_to_drop = ['SP500 30 Day Volatility', 'SPX Put Volume', 'Total SPX Options Volume', 'VIX']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# 製作 Lag 特徵
target_col = 'SP500 Log Returns'
max_lag = 7
df_original = df.copy()

for lag in range(1, max_lag + 1):
    lagged = df_original.shift(lag)
    lagged.columns = [f'{col}_lag{lag}' for col in df_original.columns]
    df = pd.concat([df, lagged], axis=1)

df = df.dropna()

# 定義 X 和 y
y = df[target_col]
feature_cols = [col for col in df.columns if '_lag' in col]
X = df[feature_cols]

print(f"Data Prepared. X Shape: {X.shape}")

# ==============================================================================
# 2. 設定 K-Fold 與 模型
# ==============================================================================
tscv = TimeSeriesSplit(n_splits=5)

feature_methods = ["Original (Scaled)", "PCA", "KPCA", "FRPCA"]

models = {
    "Linear Regression": LinearRegression(),
    "SVR":               SVR(kernel='rbf', C=1.0, epsilon=0.01),
    "Random Forest":     RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED),
    "XGBoost":           xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, 
                                          subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED),
    "LightGBM":          lgb.LGBMRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, 
                                           subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED, verbose=-1)
}

# ==============================================================================
# 3. 執行實驗 (包含 Sharpe Ratio 計算與預測值儲存)
# ==============================================================================
results_list = []
prediction_storage = {} # 用來存每一天、每一個模型的預測值

print("\n" + "="*100)
print(f"{'Feature Set':<18} | {'Model':<18} | {'MAE':<10} | {'DA':<8} | {'Sharpe':<8}")
print("="*100)

for feature_name in feature_methods:
    for model_name, model_template in models.items():
        
        # 暫存變數
        fold_scores = {'mae': [], 'rmse': [], 'da': [], 'train_mae': [], 'train_rmse': [], 'train_da': []}
        
        # 用來串接這個組合在所有 Fold 的預測值 (為了最後存檔和算 Sharpe)
        full_preds_series = pd.Series(dtype=float)
        
        for train_index, test_index in tscv.split(X):
            
            # --- A. 切分數據 ---
            X_tr_raw, X_te_raw = X.iloc[train_index], X.iloc[test_index]
            y_tr, y_te = y.iloc[train_index], y.iloc[test_index]
            
            # --- B. 標準化 ---
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr_raw)
            X_te_scaled = scaler.transform(X_te_raw)
            
            # --- C. 特徵工程 ---
            if feature_name == "Original (Scaled)":
                X_tr_final, X_te_final = X_tr_scaled, X_te_scaled
            elif feature_name == "PCA":
                pca = PCA(n_components=0.88, random_state=RANDOM_SEED)
                X_tr_final = pca.fit_transform(X_tr_scaled)
                X_te_final = pca.transform(X_te_scaled)
            elif feature_name == "KPCA":
                kpca = KernelPCA(n_components=20, kernel='rbf', gamma=0.01, 
                                 fit_inverse_transform=True, random_state=RANDOM_SEED)
                X_tr_final = kpca.fit_transform(X_tr_scaled)
                X_te_final = kpca.transform(X_te_scaled)
            elif feature_name == "FRPCA":
                mcd = MinCovDet(random_state=RANDOM_SEED).fit(X_tr_scaled)
                cov_robust = mcd.covariance_
                eigvals, eigvecs = np.linalg.eigh(cov_robust)
                k_frpca = min(20, X_tr_scaled.shape[1])
                idx = np.argsort(eigvals)[::-1]
                eigvecs_selected = eigvecs[:, idx[:k_frpca]]
                X_tr_final = X_tr_scaled.dot(eigvecs_selected)
                X_te_final = X_te_scaled.dot(eigvecs_selected)

            # --- D. 訓練與預測 ---
            # 使用 clone 確保模型重置，避免 LightGBM 警告
            current_model = clone(model_template)
            current_model.fit(X_tr_final, y_tr)
            
            y_tr_pred = current_model.predict(X_tr_final)
            y_te_pred = current_model.predict(X_te_final)
            
            # --- E. 收集預測值 (重要！) ---
            # 將這個 Fold 的預測變成 Series 並串接起來
            fold_pred_series = pd.Series(y_te_pred, index=y_te.index)
            full_preds_series = pd.concat([full_preds_series, fold_pred_series])
            
            # --- F. 計算 Fold 統計指標 ---
            fold_scores['mae'].append(mean_absolute_error(y_te, y_te_pred))
            fold_scores['rmse'].append(np.sqrt(mean_squared_error(y_te, y_te_pred)))
            fold_scores['da'].append(np.mean(np.sign(y_te_pred) == np.sign(y_te)))
            fold_scores['train_mae'].append(mean_absolute_error(y_tr, y_tr_pred))
            fold_scores['train_rmse'].append(np.sqrt(mean_squared_error(y_tr, y_tr_pred)))
            fold_scores['train_da'].append(np.mean(np.sign(y_tr_pred) == np.sign(y_tr)))

        # --- G. 迴圈結束後：計算 Sharpe Ratio ---
        # 找出對應日期的真實回報
        aligned_actuals = y.loc[full_preds_series.index]
        
        # 模擬交易：預測 > 0 買入持有，否則空手 (回報=0)
        strategy_returns = np.where(full_preds_series > 0, aligned_actuals, 0)
        
        # 計算年化 Sharpe (假設無風險利率=0, 一年252天)
        std_dev = np.std(strategy_returns)
        if std_dev == 0:
            sharpe = 0
        else:
            sharpe = (np.mean(strategy_returns) / std_dev) * np.sqrt(252)

        # --- H. 存入總表 ---
        avg_results = {k: np.mean(v) for k, v in fold_scores.items()}
        
        results_list.append({
            'Feature Method': feature_name,
            'Model Name': model_name,
            'Train MAE': avg_results['train_mae'],
            'Test MAE': avg_results['mae'],
            'Train RMSE': avg_results['train_rmse'],
            'Test RMSE': avg_results['rmse'],
            'Train DA': avg_results['train_da'],
            'Test DA': avg_results['da'],
            'Sharpe Ratio': sharpe  # 新增欄位
        })
        
        # --- I. 儲存該模型的詳細預測值 (為了之後存 CSV) ---
        col_name = f"{feature_name}_{model_name}"
        prediction_storage[col_name] = full_preds_series
        
        print(f"{feature_name:<18} | {model_name:<18} | {avg_results['mae']:<10.6f} | {avg_results['da']:<8.4f} | {sharpe:<8.4f}")

# 加入真實值到預測表 (只需加一次)
if len(prediction_storage) > 0:
    first_key = list(prediction_storage.keys())[0]
    prediction_storage['Actual'] = y.loc[prediction_storage[first_key].index]

# ==============================================================================
# 4. 存檔 (兩個 CSV)
# ==============================================================================

# 1. 儲存指標總表 (Metrics)
df_results = pd.DataFrame(results_list)
cols_order = ['Feature Method', 'Model Name', 
              'Train MAE', 'Test MAE', 
              'Train RMSE', 'Test RMSE', 
              'Train DA', 'Test DA',
              'Sharpe Ratio'] # 加上 Sharpe
df_results = df_results[cols_order]
df_results.to_csv("experiment_results_20_combinations_kfold_sharpe.csv", index=False)

# 2. 儲存每日詳細預測 (Predictions)
df_preds_daily = pd.DataFrame(prediction_storage)
df_preds_daily = df_preds_daily.sort_index() # 確保日期排序
df_preds_daily.to_csv("all_model_predictions_daily.csv")

print("\n" + "="*100)
print(f"Experiments Completed!")
print(f"1. Summary Metrics saved to: experiment_results_20_combinations_kfold_sharpe.csv")
print(f"2. Daily Predictions saved to: all_model_predictions_daily.csv")
print("="*100)

Loading Data...
Data Prepared. X Shape: (2257, 210)

Feature Set        | Model              | MAE        | DA       | Sharpe  
Original (Scaled)  | Linear Regression  | 0.018828   | 0.4878   | 0.3893  


/var/folders/wb/3mwh67wn0m55d0ly49mqht3r0000gn/T/ipykernel_62793/2793110074.py:130: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  full_preds_series = pd.concat([full_preds_series, fold_pred_series])
/var/folders/wb/3mwh67wn0m55d0ly49mqht3r0000gn/T/ipykernel_62793/2793110074.py:130: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  full_preds_series = pd.concat([full_preds_series, fold_pred_series])


Original (Scaled)  | SVR                | 0.006649   | 0.4771   | 0.3822  


/var/folders/wb/3mwh67wn0m55d0ly49mqht3r0000gn/T/ipykernel_62793/2793110074.py:130: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  full_preds_series = pd.concat([full_preds_series, fold_pred_series])


Original (Scaled)  | Random Forest      | 0.006489   | 0.5032   | 0.8182  


...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:749: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1399.495586955448744 > -1400.858191004737591). You may want to try with a higher value of support_fraction (current value: 0.780).
  warnings.warn(
/var/folders/wb/3mwh67wn0m55d0ly49mqht3r0000gn/T/ipykernel_62793/2793110074.py:130: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  full_preds_series = pd.concat([full_preds_series, fold_pred_series])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/covaria

FRPCA              | XGBoost            | 0.007899   | 0.4968   | 0.6236  


/opt/anaconda3/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:749: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1399.495586955448744 > -1400.858191004737591). You may want to try with a higher value of support_fraction (current value: 0.780).
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/var/folders/wb/3mwh67wn0m55d0ly49mqht3r0000gn/T/ipykernel_62793/2793110074.py:130: FutureWa

FRPCA              | LightGBM           | 0.006988   | 0.5133   | 0.8361  

Experiments Completed!
1. Summary Metrics saved to: experiment_results_20_combinations_kfold_sharpe.csv
2. Daily Predictions saved to: all_model_predictions_daily.csv


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
